# Advanced tutorial: Explore storm response

## Research question

**How do streamflow and turbidity respond on unusually wet days at a STREAM station?**

## `streamwq` functions used

- `stream.get_sites()`
- `stream.get_data("water_quality", ...)`
- `stream.get_data("streamflow", ...)`
- `stream.get_data("meteorology", ...)`
- `resample_timeseries()`
- `join_timeseries()`

The package intentionally does not define a universal storm event. This tutorial uses daily precipitation above the station's 95th percentile as a transparent example that researchers can replace.

In [ ]:
import streamwq as stream
from streamwq.joins import join_timeseries
from streamwq.helpers.pandas import resample_timeseries

## 1. Find a station and retrieve observations

In [ ]:
sites = stream.get_sites(variables="Turb_NTU", streamflow=True)
example_site = sites.column("STREAM_ID")[0].as_py()
start, end = "2019-01-01", "2021-12-31"
turbidity = stream.get_data(
    "water_quality",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "Turb_NTU"],
    start=start,
    end=end,
    quality="approved",
).to_pandas()
streamflow = stream.get_data(
    "streamflow",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "Q_m3s"],
    start=start,
    end=end,
    quality="approved",
).to_pandas()
rainfall = stream.get_data(
    "meteorology",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "APCP_surface"],
    start=start,
    end=end,
).to_pandas()
for dataset_name, table in [
    ("turbidity", turbidity),
    ("streamflow", streamflow),
    ("rainfall", rainfall),
]:
    if table.empty:
        raise ValueError(
            f"No {dataset_name} observations were returned for {example_site} "
            f"between {start} and {end}; try a different candidate site or period."
        )

## 2. Create an aligned daily dataset

In [ ]:
daily_turbidity = resample_timeseries(turbidity, "Turb_NTU", "daily", "max")
daily_flow = resample_timeseries(streamflow, "Q_m3s", "daily", "max")
daily_rain = resample_timeseries(rainfall, "APCP_surface", "daily", "sum")
daily = join_timeseries(daily_turbidity, daily_flow)
daily = join_timeseries(daily, daily_rain)
daily.head()

## 3. Apply a researcher-defined storm threshold

In [ ]:
rain_threshold = daily["APCP_surface"].quantile(0.95)
storm_days = daily[daily["APCP_surface"] > rain_threshold].copy()
print(f"Threshold: {rain_threshold:.2f}")
print(f"Storm days: {len(storm_days):,}")
storm_days[["DateTime", "APCP_surface", "Q_m3s", "Turb_NTU"]].head(10)

In [ ]:
daily.assign(storm=daily["APCP_surface"] > rain_threshold).groupby("storm")[["Q_m3s", "Turb_NTU"]].median()